# 06_model_catboost_v2.ipynb

## 목차
1. 환경 세팅 및 데이터 로드
2. 데이터 로드 (v2 분할 데이터)
3. 베이스라인 모델 학습
4. 하이퍼파라미터 튜닝
5. 최종 모델 학습 및 평가
6. 피처 중요도 분석
7. 결과 저장


## 1. 환경 세팅

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
import time, os

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, roc_curve, classification_report
)

plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')
pd.options.display.float_format = '{:.4f}'.format
print('임포트 완료')


## 2. 데이터 로드 (v2 분할 데이터)

In [ ]:
X_train = pd.read_csv('./data/X_train_v2.csv')
X_val   = pd.read_csv('./data/X_val_v2.csv')
y_train = pd.read_csv('./data/y_train_v2.csv').squeeze()
y_val   = pd.read_csv('./data/y_val_v2.csv').squeeze()

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}')
print(f'y_train 이탈률: {y_train.mean():.4f}')
print(f'y_val   이탈률: {y_val.mean():.4f}')


## 3. 베이스라인 모델 학습

In [ ]:
baseline_model = CatBoostClassifier(random_state=42, verbose=0)

start = time.time()
baseline_model.fit(X_train, y_train)
baseline_time = time.time() - start

b_pred  = baseline_model.predict(X_val)
b_proba = baseline_model.predict_proba(X_val)[:, 1]

print('--- CatBoost 베이스라인 성능 ---')
print(f'Accuracy : {accuracy_score(y_val, b_pred):.4f}')
print(f'AUC-ROC  : {roc_auc_score(y_val, b_proba):.4f}')
print(f'F1-Score : {f1_score(y_val, b_pred):.4f}')
print(f'학습 시간 : {baseline_time:.1f}초')


## 4. 하이퍼파라미터 튜닝

In [ ]:
param_grid = {
    'depth'         : [4, 6, 8],
    'learning_rate' : [0.05, 0.1],
    'iterations'    : [200, 300],
    'l2_leaf_reg'   : [1, 3, 5]
}
base = CatBoostClassifier(random_state=42, verbose=0)

grid_search = GridSearchCV(
    estimator=base, param_grid=param_grid,
    scoring='roc_auc', cv=3, n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
print('--- 최적 파라미터 ---')
print(best_params)
print(f'CV 최고 AUC: {grid_search.best_score_:.4f}')


## 5. 최종 모델 학습 및 평가

In [ ]:
final_model = CatBoostClassifier(
    **best_params, random_state=42, verbose=0, early_stopping_rounds=50
)
final_model.fit(X_train, y_train, eval_set=(X_val, y_val))

train_time = time.time() - start
print(f'최종 학습 완료 ({train_time:.1f}초)')


In [ ]:
y_pred  = final_model.predict(X_val)
y_proba = final_model.predict_proba(X_val)[:, 1]

final_acc = accuracy_score(y_val, y_pred)
final_auc = roc_auc_score(y_val, y_proba)
final_f1  = f1_score(y_val, y_pred)

print('--- CatBoost 최종 성능 ---')
print(f'Accuracy : {final_acc:.4f}')
print(f'AUC-ROC  : {final_auc:.4f}')
print(f'F1-Score : {final_f1:.4f}')
print(f'학습 시간 : {train_time:.1f}초')
print()
print(classification_report(y_val, y_pred, target_names=['유지(0)','이탈(1)']))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['유지(0)','이탈(1)'], yticklabels=['유지(0)','이탈(1)'])
plt.title('CatBoost Confusion Matrix')
plt.ylabel('실제값'); plt.xlabel('예측값')
plt.tight_layout(); plt.show()


In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_val, y_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'CatBoost (AUC = {final_auc:.4f})', color='seagreen', lw=2)
plt.plot([0,1],[0,1], linestyle='--', color='gray', label='Random Baseline')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('CatBoost ROC Curve')
plt.legend(); plt.tight_layout(); plt.show()


## 6. 피처 중요도 분석

In [ ]:
importance_df = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': final_model.get_feature_importance()
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 7))
sns.barplot(x='importance', y='feature', data=importance_df, palette='Greens_r')
plt.title('CatBoost Feature Importance (Top 20)')
plt.tight_layout(); plt.show()

print('--- 피처 중요도 Top 10 ---')
print(importance_df[['feature','importance']].head(10).to_string(index=False))


In [ ]:
# v1 대비 신규 피처 중요도 확인
new_features = ['freq_decay_rate','recent_price_ratio',
                'unique_articles_last_90d','diversity_decay_rate',
                'last_interval','interval_acceleration']

all_importance = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': final_model.get_feature_importance()
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('--- [NEW] 신규 피처 중요도 순위 ---')
new_imp = all_importance[all_importance['feature'].isin(new_features)].copy()
new_imp['전체 순위'] = new_imp.index + 1
print(new_imp[['전체 순위','feature','importance']].to_string(index=False))


## 7. 결과 저장

In [ ]:
os.makedirs('./results', exist_ok=True)

result = pd.DataFrame([{
    'model'     : 'CatBoost_v2',
    'accuracy'  : round(final_acc, 4),
    'auc_roc'   : round(final_auc, 4),
    'f1_score'  : round(final_f1, 4),
    'train_time': round(train_time, 1)
}])

result.to_csv('./results/catboost_result_v2.csv', index=False)
print('결과 저장 완료')
print(result.to_string(index=False))
